<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 21 · Interest-Rate and Spread Risk

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch21_interest_rate_spread_risk.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Chapter 21 turns market moves into portfolio profit and loss through first-order sensitivities. You will aggregate key-rate DV01 and CS01 buckets for a frozen portfolio of government and credit bonds, apply five named curve and spread scenarios to the unhedged book and to a simple hedge overlay, and identify which exposure bucket drives each loss. You will also verify the sign convention: positive DV01 and CS01 mean losses when yields or spreads rise. Finally, you will see why the linear loss model is additive across risk factors by construction, and where full repricing breaks that additivity.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script aggregates key-rate DV01 and CS01 by instrument type, then prices five named curve and spread scenarios for the unhedged portfolio and for an overlay that hedges 60 percent of 10-year DV01 and half of the spread risk. The recession shock produces the largest unhedged loss, -164,350, while the flattener shows a reduction of exactly zero because the overlay touches only 10-year rate and spread exposure.

In [ ]:
SCRIPT = ROOT / "code" / "ch21_interest_rate_spread_risk.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Tie each printed number to its exposure bucket and units before quoting it.

- Government bonds own the entire 2-year DV01 of 710 currency units per basis point; the credit bucket owns the whole CS01 of 1,130.
- Hand-check parallel up: 25 bp on DV01 totals of 870, 1,470, and 1,540 gives the 97,000 loss.
- The flattener reduction is exactly 0.00: the hedge only scales 10-year DV01 and CS01, so a 2-year and 5-year shock passes straight through.
- The recession shock combines a 20 bp front-end rally, a 45 bp long-end sell-off, and 80 bp of spread widening, which is why it dominates with an unhedged loss of -164,350.
- DV01 and CS01 are currency units per basis point; the scenario table is a teaching snapshot, while the 2024 US rates and inflation shock history is a separate official file.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch21_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Additive rate and spread shocks versus full repricing

A 10 million face position in a 4.5 percent semiannual bond sits at par when its all-in yield is 4.5 percent. A 25 bp government-rate shock and a 40 bp spread widening both move that same all-in yield. The linear model therefore books each component separately: DV01 times 25 plus DV01 times 40, which equals DV01 times 65 exactly. Repricing tells a slightly different story: because the price is convex in yield, the combined 65 bp move loses less than the linear estimate, and repricing the two shocks separately and adding the losses lands in between. Losses are positive amounts; shocks are in basis points.

In [ ]:
import numpy as np

times = np.arange(0.5, 5.01, 0.5)  # semiannual payment times
flows = np.full(10, 2.25)  # 4.5 percent coupon per period
flows[-1] += 100  # principal at maturity
scale = 100_000.0  # 10 million face per 100 par

def price(y):
    return float(np.sum(flows * (1 + y / 2) ** (-2 * times))) * scale

base = price(0.045)  # par bond at par
dv01 = base - price(0.0451)  # currency loss per 1 bp
linear = dv01 * 65  # additive 25 bp rate plus 40 bp spread
separate = (base - price(0.0475)) + (base - price(0.049))
joint = base - price(0.0515)  # one combined repricing
print(f"dv01: {dv01:,.2f}")
print(f"linear {linear:,.2f}  separate {separate:,.2f}  "
      f"joint {joint:,.2f}")

### What to observe

The position has a DV01 of 4,431.96 per basis point. The additive linear estimate of the combined 65 bp shock is 288,077.13; repricing the 25 bp and 40 bp shocks separately and summing gives 285,605.22; the single joint repricing gives 283,342.89. Convexity makes full-repricing losses subadditive, while the linear model is exactly additive by construction.

## Exercises

Work these with the printed tables and the chapter's frozen inputs; record units and signs with every result.

1. **Bucket audit.** State which instrument type owns the 2-year DV01 and which owns all of the CS01, with totals in currency units per basis point. *Hint:* Group the portfolio by type and sum the sensitivity columns.
2. **Hand-check one scenario.** Reproduce the parallel-up loss of 97,000 from the key-rate DV01 totals and the 25 bp shocks. *Hint:* Multiply each DV01 by its shock and add the three terms.
3. **Explain the zero.** Say why the flattener shows a reduction of exactly 0.00 while the recession shock shows the largest reduction, 86,780. *Hint:* The overlay scales only 10-year DV01 and CS01.
4. **Decompose the recession shock.** Split the -164,350 unhedged loss into a rate component and a spread component. *Hint:* Apply the 2-year, 5-year, and 10-year shocks to the matching DV01s, then the 80 bp spread shock to CS01.
5. **Design a hedge test.** Add a scenario in which 10-year yields fall 30 bp and spreads widen 60 bp, and predict the hedged P&L before computing it. *Hint:* Scale dv01_10y by 0.40 and cs01 by 0.50 as the chapter does.

For more detail, see Chapter 21 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">